In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# ---------------------------------------------------------
# 1. Load the raw data
# ---------------------------------------------------------
df = pd.read_csv("Employee-Attrition.csv")
print(f"Raw shape: {df.shape}")

# ---------------------------------------------------------
# 2. Check for missing values
# ---------------------------------------------------------
# Good news for this dataset: it usually has ZERO missing values.
# We still check, in case your copy differs.
missing = df.isnull().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print("No missing values found.")
else:
    print("Missing values found:\n", missing)
    # Example fix if you ever do find missing values:
    # numeric columns -> fill with the median
    # categorical columns -> fill with the most common value ("mode")

# ---------------------------------------------------------
# 3. Check for duplicate rows
# ---------------------------------------------------------
n_duplicates = df.duplicated().sum()
print(f"Duplicate rows: {n_duplicates}")
df = df.drop_duplicates()

# ---------------------------------------------------------
# 4. Drop columns that carry no information
# ---------------------------------------------------------
# These columns are the same for every employee, so they can't help
# predict anything. Keeping them just adds noise.
useless_columns = ["EmployeeCount", "StandardHours", "Over18"]

# EmployeeNumber is just an ID tag, not a real feature.
id_columns = ["EmployeeNumber"]

columns_to_drop = [c for c in (useless_columns + id_columns) if c in df.columns]
print(f"Dropping columns with no predictive value: {columns_to_drop}")
df = df.drop(columns=columns_to_drop)

# ---------------------------------------------------------
# 5. Convert simple Yes/No text columns into 1/0
# ---------------------------------------------------------
yes_no_columns = ["Attrition", "OverTime"]
for col in yes_no_columns:
    if col in df.columns:
        df[col] = df[col].map({"Yes": 1, "No": 0})

print("\nAttrition rate after cleaning:")
print(df["Attrition"].value_counts(normalize=True).round(3))

# ---------------------------------------------------------
# 6. One-hot encode the remaining categorical (multi-choice) columns
# ---------------------------------------------------------
# These columns have more than 2 categories (e.g. Department has 3),
# so we turn each category into its own 0/1 column.
categorical_columns = df.select_dtypes(include="object").columns.tolist()
print(f"\nCategorical columns to encode: {categorical_columns}")

df_encoded = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

# ---------------------------------------------------------
# 7. Quick sanity check
# ---------------------------------------------------------
print(f"\nFinal shape after cleaning: {df_encoded.shape}")
print(f"Any missing values left? {df_encoded.isnull().values.any()}")

# ---------------------------------------------------------
# 8. Save the clean dataset
# ---------------------------------------------------------
df_encoded.to_csv("Employee-Attrition-Clean.csv", index=False)
print("\nSaved cleaned file as Employee-Attrition-Clean.csv")


Raw shape: (1470, 35)
No missing values found.
Duplicate rows: 0
Dropping columns with no predictive value: ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber']

Attrition rate after cleaning:
Attrition
0    0.839
1    0.161
Name: proportion, dtype: float64

Categorical columns to encode: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus']

Final shape after cleaning: (1470, 45)
Any missing values left? False

Saved cleaned file as Employee-Attrition-Clean.csv
